## S01 — Import & Data Dictionary

# This notebook:
 1) loads the project config (prefers config_snapshot.json),
 2) imports the Excel dataset,
 3) matches raw columns to canonical variables using candidate lists,
 4) parses numeric values robustly,
 5) exports:
- canonical dataset (pcos_base.parquet / pcos_base.csv)
- data dictionary (data_dictionary.csv)
- column mapping (column_mapping.json)
 - parsing audit (parsing_audit.csv)
 - analysis readiness report (analysis_readiness.csv)


## Imports

In [ ]:
from __future__ import annotations

import json
import logging
import re
from pathlib import Path
from typing import Optional, Sequence, Dict, Any, List, Tuple

import numpy as np
import pandas as pd

## Loading config

In [ ]:
def load_json(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def resolve_config_path() -> Path:
    candidates = [
        Path("/content/reports/config_snapshot.json"),
        Path("/mnt/data/config_snapshot.json"),
        Path("/content/config.json"),
        Path("/mnt/data/config.json"),
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        "No config found. Expected config_snapshot.json or config.json in /content or /mnt/data."
    )


CONFIG_PATH = resolve_config_path()
CFG = load_json(CONFIG_PATH)

print("Loaded config:", str(CONFIG_PATH))
print("Input xlsx:", CFG["paths"]["input_xlsx"])
print("Sheet:", CFG["paths"]["sheet_name"])

Loaded config: /content/config.json
Input xlsx: /content/patients_results.xlsx
Sheet: Sheet1


## Catalogs and logging

In [ ]:
def ensure_dirs(cfg: dict) -> None:
    dir_keys = [
        "output_dir",
        "intermediate_dir",
        "figures_dir",
        "tables_dir",
        "models_dir",
        "reports_dir",
        "qc_dir",
        "supplementary_dir",
    ]
    for key in dir_keys:
        Path(cfg["paths"][key]).mkdir(parents=True, exist_ok=True)


ensure_dirs(CFG)


def setup_logging(cfg: dict) -> None:
    if not cfg.get("logging", {}).get("enabled", False):
        return

    log_file = Path(cfg["logging"]["log_file"])
    log_file.parent.mkdir(parents=True, exist_ok=True)

    root_logger = logging.getLogger()
    root_logger.setLevel(logging.INFO)

    if root_logger.handlers:
        has_file_handler = any(
            isinstance(h, logging.FileHandler) and Path(getattr(h, "baseFilename", "")) == log_file
            for h in root_logger.handlers
        )
        if not has_file_handler:
            file_handler = logging.FileHandler(log_file, encoding="utf-8")
            file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
            root_logger.addHandler(file_handler)
        return

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        handlers=[logging.FileHandler(log_file, encoding="utf-8"), logging.StreamHandler()],
    )


setup_logging(CFG)
logging.info("Logging initialized in S01.")
logging.info("Config path: %s", str(CONFIG_PATH))

INFO:root:Logging initialized in S01.
INFO:root:Config path: /content/config.json


## Normalization and parsing helpers

In [ ]:
_NUM_RE = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")


def normalize_colname(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).strip().lower())


def parse_numeric_cell_with_flag(
    x: object,
    range_policy: str = "mean",
    missing_tokens: Optional[Sequence[str]] = None
) -> Tuple[float, str]:
    if missing_tokens is None:
        missing_tokens = CFG.get("data_parsing", {}).get("missing_tokens", [])

    missing_tokens_lower = {str(t).lower() for t in missing_tokens}

    if x is None:
        return float("nan"), "missing"

    if isinstance(x, (int, float, np.integer, np.floating)):
        try:
            if np.isnan(x):
                return float("nan"), "missing"
        except Exception:
            pass
        return float(x), "numeric"

    s = str(x).strip()
    if s == "" or s.lower() in missing_tokens_lower:
        return float("nan"), "missing_token"

    s = s.replace(" ", "")
    s = s.replace(";", "|")

    if "|" in s:
        parts = [p for p in s.split("|") if p]
        vals = []
        for p in parts:
            p2 = p.replace(",", ".")
            if p2.startswith("<") or p2.startswith(">"):
                p2 = p2[1:]
            m = _NUM_RE.search(p2)
            if m:
                vals.append(float(m.group(0)))

        if len(vals) == 0:
            return float("nan"), "unparsable"
        if len(vals) == 1:
            return float(vals[0]), "range"

        if range_policy == "median":
            return float(np.median(vals)), "range"
        if range_policy == "first":
            return float(vals[0]), "range"
        if range_policy == "second":
            return float(vals[1]), "range"

        return float(np.mean(vals)), "range"

    s2 = s.replace(",", ".")

    if s2.startswith("<"):
        s2_num = s2[1:]
        m = _NUM_RE.search(s2_num)
        if not m:
            return float("nan"), "unparsable"
        return float(m.group(0)), "left_censored"

    if s2.startswith(">"):
        s2_num = s2[1:]
        m = _NUM_RE.search(s2_num)
        if not m:
            return float("nan"), "unparsable"
        return float(m.group(0)), "right_censored"

    m = _NUM_RE.search(s2)
    if not m:
        return float("nan"), "unparsable"

    try:
        return float(m.group(0)), "parsed"
    except Exception:
        return float("nan"), "unparsable"


def parse_numeric_cell(
    x: object,
    range_policy: str = "mean",
    missing_tokens: Optional[Sequence[str]] = None
) -> float:
    val, _flag = parse_numeric_cell_with_flag(
        x=x,
        range_policy=range_policy,
        missing_tokens=missing_tokens
    )
    return val


def coerce_numeric_series(ser: pd.Series, range_policy: str = "mean") -> pd.Series:
    return ser.apply(lambda v: parse_numeric_cell(v, range_policy=range_policy)).astype("float64")


def coerce_numeric_series_with_flags(ser: pd.Series, range_policy: str = "mean") -> pd.DataFrame:
    parsed = ser.apply(lambda v: parse_numeric_cell_with_flag(v, range_policy=range_policy))
    out = pd.DataFrame({
        "value": parsed.apply(lambda x: x[0]).astype("float64"),
        "parse_flag": parsed.apply(lambda x: x[1]).astype("string")
    })
    return out

## Column search

In [ ]:
def find_column_by_candidates(df: pd.DataFrame, candidates: Sequence[str]) -> Optional[str]:
    cols = list(df.columns)
    norm_map = {normalize_colname(c): c for c in cols}

    for cand in candidates:
        if cand in cols:
            return cand
        nc = normalize_colname(cand)
        if nc in norm_map:
            return norm_map[nc]
    return None

## Loading data

In [ ]:
input_path = Path(CFG["paths"]["input_xlsx"])
sheet_name = CFG["paths"]["sheet_name"]

if not input_path.exists():
    raise FileNotFoundError(f"Input file not found: {input_path}")

df_raw = pd.read_excel(input_path, sheet_name=sheet_name)

print("Raw shape:", df_raw.shape)
print("First columns:", list(df_raw.columns[:15]))

logging.info("Input file: %s", str(input_path))
logging.info("Raw shape: %s", str(df_raw.shape))

INFO:root:Input file: /content/patients_results.xlsx
INFO:root:Raw shape: (1300, 202)


Raw shape: (1300, 202)
First columns: ['Nr KG', 'Rok KG', 'Przyjęcie na oddział zlecający', 'Wypis z oddziału zlecającego', 'Wiek', '17 - OH progesteron (L79) (17-OHPG)', '17 OH progesteron (L79)', 'ALAT (ALT)', 'AMH (hormon anty-Mullerowski) (AMH_CP)', 'AMH-anty Mullerian Hormon (AMH)', 'APTT Czas kaolinowo-kefalinowy (APTTCZ)', 'ASO - ilościowo (ASOIL)', 'ASPAT (AST)', 'Androstendion (ANDRO)', 'Androstendion (I31)']


## Building a list of canonical requests

In [ ]:
cols_cfg = CFG["columns"]
core = cols_cfg["core_features"]

canonical_requests: List[Tuple[str, Sequence[str], bool, str]] = []


canonical_requests.append(("id", cols_cfg["id_candidates"], False, "id"))
canonical_requests.append(("age", cols_cfg["age_candidates"], False, "age"))


def add_block(block_name: str, block: Dict[str, Sequence[str]], optional_suffix: str = "_optional") -> None:
    for canonical_name, candidates in block.items():
        is_optional = canonical_name.endswith(optional_suffix)
        canonical = canonical_name.replace(optional_suffix, "")
        canonical_requests.append((canonical, candidates, is_optional, block_name))


add_block("thyroid", core.get("thyroid", {}))
add_block("lipids", core.get("lipids", {}))
add_block("glucose", core.get("glucose", {}))
add_block("androgens_optional", core.get("androgens_optional", {}))
add_block("insulin_optional", core.get("insulin_optional", {}))
add_block("medications_optional", core.get("medications_optional", {}))

print("Total canonical requests:", len(canonical_requests))

Total canonical requests: 25


## Raw -> canonical mapping

In [ ]:
mapping: Dict[str, Dict[str, Any]] = {}

for canonical, candidates, is_optional, source_block in canonical_requests:
    raw_col = find_column_by_candidates(df_raw, candidates)
    mapping[canonical] = {
        "raw_column": raw_col,
        "is_optional": bool(is_optional),
        "source_block": source_block,
        "candidates": list(candidates),
        "matched": raw_col is not None
    }

matched = [k for k, v in mapping.items() if v["matched"]]
missing_required = [k for k, v in mapping.items() if (not v["is_optional"]) and (not v["matched"])]

print(f"Matched {len(matched)}/{len(mapping)} canonical variables.")
if missing_required:
    print("Missing REQUIRED canonicals:", missing_required)
else:
    print("All required canonicals were matched.")

logging.info("Matched %s/%s canonical variables.", len(matched), len(mapping))
logging.info("Missing required canonicals: %s", missing_required if missing_required else "None")

INFO:root:Matched 22/25 canonical variables.
INFO:root:Missing required canonicals: None


Matched 22/25 canonical variables.
All required canonicals were matched.


## Building a canonical dataframe

In [ ]:
data = {}

for canonical, info in mapping.items():
    raw_col = info["raw_column"]
    if raw_col is None:
        data[canonical] = pd.Series([np.nan] * len(df_raw))
    else:
        data[canonical] = df_raw[raw_col]

df_can = pd.DataFrame(data)

print("Canonical shape before parsing:", df_can.shape)
df_can.head(3)

Canonical shape before parsing: (1300, 25)


,id,age,anti_tpo,anti_tg,tsh,ft4,ft3,tg,hdl,tc,...,shbg,dheas,andro,amh,lh,fsh,ins0,ins120,insulin,lt4_use
0,7611,25,"13,80",NaN,"0,969","1,20",NaN,"116,00","56,60","188,0",...,NaN,"379,0","2,91",NaN,"26,80 | 40,40","10,10",NaN,NaN,"11,90",NaN
1,8133,25,"12,60",NaN,"2,050","1,18",NaN,"144,00","41,90","196,0",...,NaN,"340,0","0,72",NaN,"3,08","5,45",NaN,NaN,"10,20",NaN
2,11028,25,"150,00",NaN,"2,500","1,29",NaN,"35,80","62,70","133,0",...,NaN,"462,0","1,72",NaN,"5,65","7,03",NaN,NaN,"20,70",NaN


## Numeric parsing + parse audit

In [ ]:
range_policy = CFG["data_parsing"]["range_notation_handling"]["default_policy"]
missing_tokens = CFG["data_parsing"]["missing_tokens"]

parse_audit_rows = []

numeric_candidates = [c for c in df_can.columns if c not in ["id"]]

for c in numeric_candidates:
    ser = df_can[c]

    parsed_df = coerce_numeric_series_with_flags(ser, range_policy=range_policy)
    df_can[c] = parsed_df["value"]

    flag_counts = (
        parsed_df["parse_flag"]
        .value_counts(dropna=False)
        .rename_axis("parse_flag")
        .reset_index(name="n")
    )
    flag_counts["canonical"] = c
    flag_counts["pct"] = (flag_counts["n"] / len(parsed_df) * 100).round(2)

    parse_audit_rows.append(flag_counts)

df_can["id"] = df_can["id"].astype("string")

parse_audit = pd.concat(parse_audit_rows, axis=0, ignore_index=True)

print("Canonical shape after parsing:", df_can.shape)
df_can.head(3)

Canonical shape after parsing: (1300, 25)


,id,age,anti_tpo,anti_tg,tsh,ft4,ft3,tg,hdl,tc,...,shbg,dheas,andro,amh,lh,fsh,ins0,ins120,insulin,lt4_use
0,7611,25.0,13.8,NaN,0.969,1.20,NaN,116.0,56.6,188.0,...,NaN,379.0,2.91,NaN,33.60,10.10,NaN,NaN,11.9,NaN
1,8133,25.0,12.6,NaN,2.050,1.18,NaN,144.0,41.9,196.0,...,NaN,340.0,0.72,NaN,3.08,5.45,NaN,NaN,10.2,NaN
2,11028,25.0,150.0,NaN,2.500,1.29,NaN,35.8,62.7,133.0,...,NaN,462.0,1.72,NaN,5.65,7.03,NaN,NaN,20.7,NaN


## Minimal validation and derived readiness

In [ ]:
def has_nonmissing(df: pd.DataFrame, cols: List[str]) -> bool:
    return all(col in df.columns and df[col].notna().any() for col in cols)


readiness_rows = []

eligibility = CFG.get("eligibility", {})

for analysis_name, required_cols in eligibility.items():
    readiness_rows.append({
        "analysis_name": analysis_name,
        "required_columns": ", ".join(required_cols),
        "all_columns_present": all(col in df_can.columns for col in required_cols),
        "all_columns_have_any_nonmissing": has_nonmissing(df_can, required_cols),
        "n_complete_cases": int(df_can[required_cols].dropna().shape[0]) if all(col in df_can.columns for col in required_cols) else 0,
    })


lt4_present = "lt4_use" in df_can.columns and df_can["lt4_use"].notna().any()
anti_tg_present = "anti_tg" in df_can.columns and df_can["anti_tg"].notna().any()
ft4_present = "ft4" in df_can.columns and df_can["ft4"].notna().any()
tsh_present = "tsh" in df_can.columns and df_can["tsh"].notna().any()

readiness_rows.extend([
    {
        "analysis_name": "lt4_sensitivity_possible",
        "required_columns": "lt4_use",
        "all_columns_present": "lt4_use" in df_can.columns,
        "all_columns_have_any_nonmissing": bool(lt4_present),
        "n_complete_cases": int(df_can[["lt4_use"]].dropna().shape[0]) if "lt4_use" in df_can.columns else 0,
    },
    {
        "analysis_name": "anti_tg_sensitivity_possible",
        "required_columns": "anti_tg",
        "all_columns_present": "anti_tg" in df_can.columns,
        "all_columns_have_any_nonmissing": bool(anti_tg_present),
        "n_complete_cases": int(df_can[["anti_tg"]].dropna().shape[0]) if "anti_tg" in df_can.columns else 0,
    },
    {
        "analysis_name": "strict_euthyroid_possible",
        "required_columns": "tsh, ft4",
        "all_columns_present": all(col in df_can.columns for col in ["tsh", "ft4"]),
        "all_columns_have_any_nonmissing": bool(tsh_present and ft4_present),
        "n_complete_cases": int(df_can[["tsh", "ft4"]].dropna().shape[0]) if all(col in df_can.columns for col in ["tsh", "ft4"]) else 0,
    },
])

analysis_readiness = pd.DataFrame(readiness_rows)
analysis_readiness

,analysis_name,required_columns,all_columns_present,all_columns_have_any_nonmissing,n_complete_cases
0,required_for_primary,"age, anti_tpo, tg, hdl",True,True,1053
1,required_for_secondary_non_hdl,"age, anti_tpo, tc, hdl",True,True,1053
2,required_for_secondary_ogtt120,"age, anti_tpo, glu120",True,True,1035
3,lt4_sensitivity_possible,lt4_use,True,False,0
4,anti_tg_sensitivity_possible,anti_tg,True,True,257
5,strict_euthyroid_possible,"tsh, ft4",True,True,1181


## Data dictionary

In [ ]:
def pct_missing(s: pd.Series) -> float:
    return float((s.isna().mean() * 100).round(2))


def sample_values(s: pd.Series, n: int = 5) -> List[str]:
    vals = s.dropna().astype(str).unique().tolist()
    return vals[:n]


rows = []
for canonical, info in mapping.items():
    raw_col = info["raw_column"]
    s = df_can[canonical]

    rows.append({
        "canonical": canonical,
        "raw_column": raw_col,
        "matched": info["matched"],
        "is_optional": info["is_optional"],
        "source_block": info["source_block"],
        "dtype": str(s.dtype),
        "n_missing": int(s.isna().sum()),
        "pct_missing": pct_missing(s),
        "n_unique_non_missing": int(s.dropna().nunique()),
        "example_values": "; ".join(sample_values(s, n=5)),
    })

dd = pd.DataFrame(rows).sort_values(
    ["matched", "is_optional", "pct_missing"],
    ascending=[False, True, True]
)

dd.head(15)

,canonical,raw_column,matched,is_optional,source_block,dtype,n_missing,pct_missing,n_unique_non_missing,example_values
0,id,Nr KG,True,False,id,string,0,0.00,1300,7611; 8133; 11028; 12103; 12696
1,age,Wiek,True,False,age,float64,0,0.00,10,25.0; 24.0; 23.0; 22.0; 21.0
7,tg,Triglicerydy (TG),True,False,lipids,float64,117,9.00,559,116.0; 144.0; 35.8; 73.3; 149.0
8,hdl,HDL cholesterol (HDL),True,False,lipids,float64,117,9.00,463,56.6; 41.9; 62.7; 58.1; 39.0
4,tsh,TSH (TSH),True,False,thyroid,float64,118,9.08,420,0.969; 2.05; 2.5; 1.2; 1.05
2,anti_tpo,Anty-TPO (ATA_TPO),True,False,thyroid,float64,245,18.85,307,13.8; 12.6; 150.0; 9.0; 89.1
16,dheas,DHEAS (DHEA),True,True,androgens_optional,float64,115,8.85,607,379.0; 340.0; 462.0; 310.0; 166.0
9,tc,Cholesterol całkowity (TCHOL),True,True,lipids,float64,117,9.00,155,188.0; 196.0; 133.0; 147.0; 150.0
10,ldl,LDL cholesterol (LDL),True,True,lipids,float64,117,9.00,926,108.2; 125.3; 63.14; 74.24; 81.2
19,lh,LH (LH),True,True,androgens_optional,float64,118,9.08,704,33.6; 3.08; 5.65; 11.1; 9.46


## Missing and Critical Variables Report

In [ ]:
critical_vars = ["id", "age", "anti_tpo", "tsh", "tg", "hdl"]
critical_status = []

for var in critical_vars:
    info = mapping.get(var, {})
    critical_status.append({
        "canonical": var,
        "matched": bool(info.get("matched", False)),
        "raw_column": info.get("raw_column", None),
        "n_nonmissing": int(df_can[var].notna().sum()) if var in df_can.columns else 0,
        "pct_missing": float((df_can[var].isna().mean() * 100).round(2)) if var in df_can.columns else 100.0,
    })

critical_status_df = pd.DataFrame(critical_status)
critical_status_df

,canonical,matched,raw_column,n_nonmissing,pct_missing
0,id,True,Nr KG,1300,0.00
1,age,True,Wiek,1300,0.00
2,anti_tpo,True,Anty-TPO (ATA_TPO),1055,18.85
3,tsh,True,TSH (TSH),1182,9.08
4,tg,True,Triglicerydy (TG),1183,9.00
5,hdl,True,HDL cholesterol (HDL),1183,9.00


## Exporting artifacts

In [ ]:
out_dd = Path(CFG["paths"]["tables_dir"]) / "data_dictionary.csv"
out_map = Path(CFG["paths"]["intermediate_dir"]) / "column_mapping.json"
out_parquet = Path(CFG["paths"]["intermediate_dir"]) / "pcos_base.parquet"
out_csv = Path(CFG["paths"]["intermediate_dir"]) / "pcos_base.csv"
out_parse_audit = Path(CFG["paths"]["qc_dir"]) / "parsing_audit.csv"
out_readiness = Path(CFG["paths"]["reports_dir"]) / "analysis_readiness.csv"
out_critical = Path(CFG["paths"]["reports_dir"]) / "critical_variable_status.csv"

dd.to_csv(out_dd, index=False)
parse_audit.to_csv(out_parse_audit, index=False)
analysis_readiness.to_csv(out_readiness, index=False)
critical_status_df.to_csv(out_critical, index=False)

with open(out_map, "w", encoding="utf-8") as f:
    json.dump(mapping, f, ensure_ascii=False, indent=2)

try:
    df_can.to_parquet(out_parquet, index=False)
    print("Saved canonical dataset (parquet):", out_parquet)
except Exception as e:
    print("Parquet save failed, reason:", repr(e))
    logging.warning("Parquet save failed: %r", e)

df_can.to_csv(out_csv, index=False)

print("Saved canonical dataset (csv):", out_csv)
print("Saved data dictionary:", out_dd)
print("Saved column mapping:", out_map)
print("Saved parsing audit:", out_parse_audit)
print("Saved analysis readiness:", out_readiness)
print("Saved critical variable status:", out_critical)

Saved canonical dataset (parquet): /content/outputs/intermediate/pcos_base.parquet
Saved canonical dataset (csv): /content/outputs/intermediate/pcos_base.csv
Saved data dictionary: /content/outputs/tables/data_dictionary.csv
Saved column mapping: /content/outputs/intermediate/column_mapping.json
Saved parsing audit: /content/outputs/qc/parsing_audit.csv
Saved analysis readiness: /content/reports/analysis_readiness.csv
Saved critical variable status: /content/reports/critical_variable_status.csv


## Summary

In [ ]:
print("\nS01 summary")
print("-----------")
print(f"Loaded raw dataset: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
print(f"Canonical dataset:  {df_can.shape[0]} rows x {df_can.shape[1]} columns")
print(f"Matched canonicals: {len(matched)}/{len(mapping)}")

if missing_required:
    print("Missing REQUIRED canonicals:", missing_required)
else:
    print("All required canonicals were matched.")

primary_ready = analysis_readiness.loc[
    analysis_readiness["analysis_name"] == "required_for_primary",
    "n_complete_cases"
]
if len(primary_ready) > 0:
    print("Complete cases for primary endpoint readiness:", int(primary_ready.iloc[0]))

lt4_row = analysis_readiness.loc[analysis_readiness["analysis_name"] == "lt4_sensitivity_possible"]
if len(lt4_row) > 0:
    print("LT4 sensitivity possible:", bool(lt4_row["all_columns_have_any_nonmissing"].iloc[0]))

anti_tg_row = analysis_readiness.loc[analysis_readiness["analysis_name"] == "anti_tg_sensitivity_possible"]
if len(anti_tg_row) > 0:
    print("anti-TG sensitivity possible:", bool(anti_tg_row["all_columns_have_any_nonmissing"].iloc[0]))


S01 summary
-----------
Loaded raw dataset: 1300 rows x 202 columns
Canonical dataset:  1300 rows x 25 columns
Matched canonicals: 22/25
All required canonicals were matched.
Complete cases for primary endpoint readiness: 1053
LT4 sensitivity possible: False
anti-TG sensitivity possible: True


## Completion checklist

## S01 completion checklist

 - Confirm required canonical variables were matched:
 at minimum id, age, anti_tpo, tsh, tg, hdl
- Inspect data_dictionary.csv for wrong matches or unexpected missingness
- Inspect parsing_audit.csv for excessive unparsable values
- Inspect analysis_readiness.csv:
  * primary endpoint possible?
  * non-HDL possible?
  * OGTT120 possible?
  * LT4 sensitivity possible?
  * anti-TG sensitivity possible?
- Proceed to S02 (QC & transforms)